In [ ]:
import os
from datetime import datetime                          # ← Manquait dans l'original

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Imports nécessaires pour les tools
from langchain_huggingface import HuggingFaceEmbeddings   # ← pip install langchain-huggingface
from langchain_chroma import Chroma                        # ← pip install langchain-chroma

PERSIST_DIR = "/workspace/data/chroma_db_demo"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")

print("✅ Imports OK")

In [ ]:
from langchain_core.tools import tool

# ============================================================
# DÉFINITION DES TOOLS
# Chaque @tool expose une fonction Python à l'agent.
# La docstring est CRITIQUE : c'est ce que le LLM lit pour
# décider quand et comment appeler le tool.
# ============================================================

@tool
def creer_ticket(titre: str, categorie: str, priorite: str) -> str:
    """Crée un ticket dans le système de ticketing (simulé).
    Paramètres :
    - titre : résumé court du problème
    - categorie : 'materiel', 'logiciel', 'reseau', 'compte'
    - priorite : 'P1' (critique), 'P2' (haute), 'P3' (moyenne), 'P4' (basse)
    Retourne l'ID du ticket créé."""
    ticket_id = f"TICK-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    print(f"\n  [TICKETING] {ticket_id} | {priorite} | {categorie} | {titre}\n")
    return f"Ticket {ticket_id} créé avec priorité {priorite}, catégorie {categorie}."


@tool
def chercher_kb(question: str) -> str:
    """Cherche dans la base de connaissances support et retourne les passages
    les plus pertinents. À utiliser pour répondre à des questions techniques."""
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    vs = Chroma(
        collection_name="demo_kb",
        embedding_function=embeddings,
        persist_directory=PERSIST_DIR,
    )
    docs = vs.similarity_search(question, k=2)
    if not docs:
        return "Aucune information pertinente trouvée dans la KB."
    return "\n\n---\n\n".join(d.page_content for d in docs)


@tool
def heure() -> str:
    """Retourne la date et l'heure actuelles. Sans paramètre."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


TOOLS = [creer_ticket, chercher_kb, heure]

print(f"✅ {len(TOOLS)} tools enregistrés : {[t.name for t in TOOLS]}")

✅ 3 tools enregistrés : ['creer_ticket', 'chercher_kb', 'heure']


In [ ]:
# ============================================================
# CONSTRUCTION DE L'AGENT — API LangGraph V1.0+
#
# Historique des migrations :
#   LangChain < 0.3  : from langchain.agents import AgentExecutor, create_tool_calling_agent
#   LangGraph 0.x    : from langgraph.prebuilt import create_react_agent
#   LangGraph 1.0+   : from langchain.agents import create_react_agent  ← ACTUEL
#
# Leçon : les imports LangChain/LangGraph bougent vite.
# En cas de doute : pip show langgraph && chercher le message de dépréciation.
# ============================================================

from langchain.agents import create_agent   # ← Import correct pour LangGraph V1.0+

SYSTEM_PROMPT = (
    "Tu es l'assistant support technique de l'entreprise. Tu peux :\n"
    "- chercher dans la base de connaissances pour répondre à des questions techniques,\n"
    "- créer un ticket de support quand l'utilisateur le demande,\n"
    "- retourner l'heure actuelle.\n\n"
    "Sois concis. Quand tu crées un ticket, propose toujours une priorité justifiée."
)

def build_agent():
    llm = ChatOllama(model="llama3.2:3b", base_url=OLLAMA_HOST, temperature=0)

    agent = create_agent(
        model=llm,
        tools=TOOLS,
        system_prompt=SYSTEM_PROMPT,
    )
    return agent

print("✅ Fonction build_agent() prête (API LangGraph moderne)")

✅ Fonction build_agent() prête (API LangGraph moderne)


In [ ]:
executor = build_agent()

question = (
    "Mon Outlook plante au démarrage avec un message de profil corrompu. "
    "Cherche d'abord dans la KB ce qu'il faut faire, puis crée un ticket P3 "
    "catégorie 'logiciel' si la solution ne fonctionne pas."
)

result = executor.invoke(
    {"messages": [("human", question)]} # type: ignore
)

reponse_finale = result["messages"][-1].content

print("\n" + "=" * 60)
print("RÉPONSE FINALE :")
print(reponse_finale)
print("=" * 60)

print("\n Trace complète de l'agent :")
for i, msg in enumerate(result["messages"]):
    role = type(msg).__name__.replace("Message", "")
    contenu = str(msg.content)[:200] + "..." if len(str(msg.content)) > 200 else str(msg.content)
    print(f"  [{i}] {role:10s} : {contenu}")


  [TICKETING] TICK-20260528212733 | P3 | logiciel | Problème d’Outlook


RÉPONSE FINALE :
Merci de confirmer ! Le ticket a été créé avec les informations suivantes :

**Titre du ticket :** Problème d’Outlook
**Catégorie :** Logiciel
**Priorité :** P3
**Description du problème :** Mon Outlook plante au démarrage avec un message de profil corrompu.

Je vais maintenant attendre votre réponse pour savoir si la solution proposée a fonctionné ou non.

📋 Trace complète de l'agent :
  [0] Human      : Mon Outlook plante au démarrage avec un message de profil corrompu. Cherche d'abord dans la KB ce qu'il faut faire, puis crée un ticket P3 catégorie 'logiciel' si la solution ne fonctionne pas.
  [1] AI         : 
  [2] Tool       : Logiciels — Outlook, Office 365, ERP

Outlook plante au démarrage — "Profil corrompu"

Symptôme : Outlook se ferme brutalement au démarrage, ou affiche le message "Le profil Outlook est corrompu" ou "...
  [3] Tool       : Ticket TICK-20260528212733 créé avec priorit

---
### Approche 2 — `interrupt()` LangGraph (version production)

Ici, le graph se met **réellement en pause** entre deux tours, comme un checkpoint.
Deux ingrédients nécessaires :

1. **`MemorySaver`** : un checkpointer qui sauvegarde l'état du graph entre les appels.
2. **`thread_id`** : un identifiant de conversation, pour retrouver l'état sauvegardé.

Le graph s'arrête sur `interrupt()`, retourne le contrôle à Python, et reprend là où il s'était arrêté quand on le relance avec le même `thread_id`.

In [ ]:
# ============================================================
# APPROCHE 2 — interrupt() LANGGRAPH (production-ready)
# ============================================================
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

# MemorySaver stocke l'état du graph en mémoire (RAM)
# En production on utiliserait SqliteSaver ou PostgresSaver
memory = MemorySaver()

# On définit un tool spécial qui déclenche la pause de l'agent
# Ce tool appelle interrupt() pour suspendre l'exécution du graph
@tool
def demander_confirmation_utilisateur(question: str) -> str:
    """Pose une question à l'utilisateur et attend sa réponse avant de continuer.
    Utilise ce tool quand tu as besoin d'une validation humaine pour poursuivre.
    Paramètre : question — ce que tu veux demander à l'utilisateur."""
    # interrupt() suspend le graph ici et retourne 'question' comme valeur
    # L'exécution reprendra depuis ce point quand on appellera Command(resume=...)
    reponse = interrupt(question)   # ← le graph se met en pause ici
    return reponse                  # ← reponse = ce que l'utilisateur a saisi


# Construction de l'agent avec le checkpointer ET le tool de confirmation
llm = ChatOllama(model="gemma3:4b", base_url=OLLAMA_HOST, temperature=0)

TOOLS_AVEC_CONFIRM = TOOLS + [demander_confirmation_utilisateur]

agent_interactif = create_agent(
    model=llm,
    tools=TOOLS_AVEC_CONFIRM,
    system_prompt=(
        SYSTEM_PROMPT +
        "\n- Avant de créer un ticket, utilise TOUJOURS le tool "
        "'demander_confirmation_utilisateur' pour vérifier que la solution KB a échoué."
    ),
    checkpointer=memory,            # ← active la persistance d'état entre les appels
)

# Chaque conversation a un thread_id unique — comme un ID de session
config = {"configurable": {"thread_id": "session-demo-001"}}

print("\u2705 Agent interactif prêt (avec interrupt + MemorySaver)")

NameError: name 'ChatOllama' is not defined

In [ ]:
# ============================================================
# EXÉCUTION DU GRAPH AVEC PAUSE
#
# invoke() retourne dès que interrupt() est appelé.
# La valeur de retour contient '__interrupt__' avec la question posée.
# ============================================================

print("=" * 60)
print("TOUR 1 — L'agent cherche dans la KB et pose sa question")
print("=" * 60)

question = (
    "Mon Outlook plante au démarrage avec un profil corrompu. "
    "Cherche dans la KB, propose la solution, puis demande-moi si ça a fonctionné "
    "avant de créer un ticket."
)

# Premier appel — le graph va s'arrêter sur interrupt()
etat = agent_interactif.invoke(
    {"messages": [("human", question)]}, # type: ignore
    config=config,                  # ← thread_id pour retrouver l'état plus tard # type: ignore
)

# Quand interrupt() est déclenché, la clé '__interrupt__' est présente dans l'état
if "__interrupt__" in etat:
    question_agent = etat["__interrupt__"][0].value
    print("\nGRAPH EN PAUSE — L'agent attend ta réponse :")
    print(f"Agent : {question_agent}\n")
else:
    # L'agent a répondu sans déclencher interrupt (il n'a pas jugé nécessaire)
    print(f"\nAgent (réponse directe) : {etat['messages'][-1].content}")